In [ ]:
"""
MODEL A: ANSWER VERIFICATION — BULLETPROOF CROSS-SESSION PIPELINE
================================================================
- Mount Google Drive FIRST (run the cell below before this)
- All checkpoints save to Drive so crashes don't lose progress
- Re-running resumes from where it left off automatically

CELL 0 — Run this alone first:
    from google.colab import drive
    drive.mount('/content/drive')

Then run this file.
"""

import pandas as pd
import numpy as np
import pickle
import os
import json
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION  ← only edit things here
# ============================================================================

class Config:
    DATA_FILE        = 'dev.csv'

    # ✅ Clean Drive folder (NOT .ipynb_checkpoints)
    CHECKPOINT_DIR   = '/content/drive/MyDrive/.ipynb_checkpoints'

    RANDOM_SEED      = 42
    TRAIN_SIZE       = 0.70
    VAL_SIZE         = 0.15
    TEST_SIZE        = 0.15

    TFIDF_MAX_FEATURES = 300
    TFIDF_NGRAM_RANGE  = (1, 2)
    TFIDF_MIN_DF       = 2
    TFIDF_MAX_DF       = 0.95

    REQUIRED_COLUMNS = ['article', 'question', 'A', 'B', 'C', 'D', 'answer']
    VALID_ANSWERS    = ['A', 'B', 'C', 'D']


# ============================================================================
# DRIVE MOUNT GUARD
# ============================================================================

def ensure_drive_mounted():
    """Mount Drive if not already mounted, with a clear error if it fails."""
    drive_root = '/content/drive/MyDrive'
    if os.path.isdir(drive_root):
        print("✓ Google Drive already mounted")
        return True
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✓ Google Drive mounted")
        return True
    except Exception as e:
        print(f"""
❌ Could not mount Google Drive: {e}

Please run this in a separate cell FIRST:
    from google.colab import drive
    drive.mount('/content/drive')

Then re-run this script.
""")
        return False


# ============================================================================
# CHECKPOINT MANAGER
# ============================================================================

class CheckpointManager:
    """Save/load objects to Google Drive. Survives Colab crashes."""

    def __init__(self, checkpoint_dir: str):
        self.checkpoint_dir = checkpoint_dir
        Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
        print(f"✓ Checkpoint dir: {checkpoint_dir}")

    # ── internal helpers ──────────────────────────────────────────────────

    def _path(self, name: str) -> str:
        return os.path.join(self.checkpoint_dir, name)

    # ── public API ────────────────────────────────────────────────────────

    def save(self, obj, name: str, verbose=True) -> bool:
        try:
            fp = self._path(name)
            with open(fp, 'wb') as f:
                pickle.dump(obj, f)
            size_mb = os.path.getsize(fp) / (1024 ** 2)
            if verbose:
                print(f"  ✓ Saved {name}  ({size_mb:.2f} MB)")
            return True
        except Exception as e:
            print(f"  ✗ Save failed [{name}]: {e}")
            return False

    def load(self, name: str, verbose=True):
        fp = self._path(name)
        if not os.path.exists(fp):
            if verbose:
                print(f"  ⚠  Not found: {name}")
            return None
        try:
            with open(fp, 'rb') as f:
                obj = pickle.load(f)
            if verbose:
                print(f"  ✓ Loaded {name}")
            return obj
        except Exception as e:
            print(f"  ✗ Load failed [{name}]: {e}")
            return None

    def exists(self, name: str) -> bool:
        return os.path.exists(self._path(name))

    def list_all(self) -> list:
        if not os.path.exists(self.checkpoint_dir):
            return []
        return sorted(os.listdir(self.checkpoint_dir))

    def status(self):
        """Print what's already saved."""
        files = self.list_all()
        print(f"\nCheckpoint directory: {self.checkpoint_dir}")
        print(f"Files saved ({len(files)}):")
        for f in files:
            size = os.path.getsize(self._path(f)) / (1024 ** 2)
            print(f"  {f:<35} {size:.2f} MB")


# ============================================================================
# STEP 1 — LOAD DATA
# ============================================================================

def load_data(cp: CheckpointManager) -> pd.DataFrame:
    print("\n" + "=" * 70)
    print("STEP 1: LOAD DATA")
    print("=" * 70)

    # Resume
    if cp.exists('df_clean.pkl'):
        df = cp.load('df_clean.pkl')
        if df is not None and len(df) > 0:
            print(f"✓ Resumed from checkpoint — {len(df)} rows")
            return df

    # Try multiple loading strategies
    strategies = [
        ('utf-8',                    lambda: pd.read_csv(Config.DATA_FILE, encoding='utf-8')),
        ('latin-1',                  lambda: pd.read_csv(Config.DATA_FILE, encoding='latin1')),
        ('utf-8 + skip bad lines',   lambda: pd.read_csv(Config.DATA_FILE, encoding='utf-8',  on_bad_lines='skip')),
        ('latin-1 + skip bad lines', lambda: pd.read_csv(Config.DATA_FILE, encoding='latin1', on_bad_lines='skip')),
        ('python engine',            lambda: pd.read_csv(Config.DATA_FILE, engine='python',   on_bad_lines='skip')),
    ]

    df = None
    for label, fn in strategies:
        try:
            print(f"  Trying [{label}] ...", end=" ")
            df = fn()
            print(f"✓  {len(df)} rows")
            break
        except Exception as e:
            print(f"✗  {str(e)[:60]}")

    if df is None or len(df) == 0:
        raise RuntimeError("All CSV loading strategies failed. Check your file path.")

    # Validate columns
    missing = [c for c in Config.REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Found: {list(df.columns)}")

    print(f"\n  Raw rows     : {len(df)}")
    print(f"  Answer dist  :\n{df['answer'].value_counts().to_string()}")

    # Clean
    df = df.dropna(subset=Config.REQUIRED_COLUMNS)
    df = df[df['answer'].isin(Config.VALID_ANSWERS)].reset_index(drop=True)
    print(f"\n✓ After cleaning: {len(df)} rows")

    cp.save(df, 'df_clean.pkl', verbose=False)
    return df


# ============================================================================
# STEP 2 — SPLIT DATA
# ============================================================================

def split_data(df: pd.DataFrame, cp: CheckpointManager):
    print("\n" + "=" * 70)
    print("STEP 2: SPLIT DATA  (70 / 15 / 15)")
    print("=" * 70)

    # Resume
    if cp.exists('split_train.pkl') and cp.exists('split_val.pkl') and cp.exists('split_test.pkl'):
        train_df = cp.load('split_train.pkl')
        val_df   = cp.load('split_val.pkl')
        test_df  = cp.load('split_test.pkl')
        print(f"✓ Resumed — train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")
        return train_df, val_df, test_df

    df = df.sample(frac=1, random_state=Config.RANDOM_SEED).reset_index(drop=True)
    n = len(df)
    t1 = int(n * Config.TRAIN_SIZE)
    t2 = int(n * (Config.TRAIN_SIZE + Config.VAL_SIZE))

    train_df = df[:t1].reset_index(drop=True)
    val_df   = df[t1:t2].reset_index(drop=True)
    test_df  = df[t2:].reset_index(drop=True)

    assert len(train_df) + len(val_df) + len(test_df) == n, "Split mismatch!"

    print(f"✓ Train : {len(train_df):6d}  ({100*len(train_df)/n:.1f}%)")
    print(f"✓ Val   : {len(val_df):6d}  ({100*len(val_df)/n:.1f}%)")
    print(f"✓ Test  : {len(test_df):6d}  ({100*len(test_df)/n:.1f}%)")

    cp.save(train_df, 'split_train.pkl', verbose=False)
    cp.save(val_df,   'split_val.pkl',   verbose=False)
    cp.save(test_df,  'split_test.pkl',  verbose=False)

    return train_df, val_df, test_df


# ============================================================================
# STEP 3 — FEATURE ENGINEERING
# ============================================================================

def _make_text(article, question, option) -> str:
    return (f"{str(article).lower().strip()} "
            f"{str(question).lower().strip()} "
            f"{str(option).lower().strip()}")


def _lexical_features(article, question, option) -> np.ndarray:
    a = set(str(article).lower().split())
    q = set(str(question).lower().split())
    o = set(str(option).lower().split())
    return np.array([
        len(o),
        len(o & q),
        len(o & a),
        len(o | q | a),
    ], dtype=float)


def _build_xy(df: pd.DataFrame, vectorizer, skipped_log=True):
    X, y, skipped = [], [], 0

    for _, row in df.iterrows():
        try:
            article  = row['article']
            question = row['question']
            correct  = str(row['answer']).upper()

            for opt in ['A', 'B', 'C', 'D']:
                try:
                    text     = _make_text(article, question, row[opt])
                    tfidf    = vectorizer.transform([text]).toarray()[0]
                    lex      = _lexical_features(article, question, row[opt])
                    X.append(np.concatenate([tfidf, lex]))
                    y.append(1 if opt == correct else 0)
                except Exception:
                    skipped += 1
        except Exception:
            skipped += 1

    if skipped_log and skipped:
        print(f"  ⚠  Skipped {skipped} malformed samples")

    return np.array(X), np.array(y)


def preprocess_all(train_df, val_df, test_df, cp: CheckpointManager):
    print("\n" + "=" * 70)
    print("STEP 3: FEATURE ENGINEERING")
    print("=" * 70)

    # Full resume — all matrices already exist
    keys = ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test',
            'vectorizer', 'scaler']
    if all(cp.exists(k + '.pkl') for k in keys):
        print("✓ Resumed all feature matrices from checkpoint")
        loaded = {k: cp.load(k + '.pkl', verbose=False) for k in keys}
        return (loaded['X_train'], loaded['y_train'],
                loaded['X_val'],   loaded['y_val'],
                loaded['X_test'],  loaded['y_test'],
                loaded['vectorizer'], loaded['scaler'])

    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import StandardScaler

    # ── Fit vectorizer ────────────────────────────────────────────────────
    print("\n[1/3] Fitting TF-IDF on training data...")
    if cp.exists('vectorizer.pkl'):
        vectorizer = cp.load('vectorizer.pkl')
    else:
        vectorizer = TfidfVectorizer(
            max_features = Config.TFIDF_MAX_FEATURES,
            stop_words   = 'english',
            ngram_range  = Config.TFIDF_NGRAM_RANGE,
            min_df       = Config.TFIDF_MIN_DF,
            max_df       = Config.TFIDF_MAX_DF,
        )
        all_texts = []
        for _, row in train_df.iterrows():
            for opt in ['A', 'B', 'C', 'D']:
                try:
                    all_texts.append(_make_text(row['article'], row['question'], row[opt]))
                except Exception:
                    pass
        vectorizer.fit(all_texts)
        cp.save(vectorizer, 'vectorizer.pkl')
        print(f"  ✓ Vocabulary size: {len(vectorizer.get_feature_names_out())}")

    # ── Training features ─────────────────────────────────────────────────
    print("\n[2/3] Building feature matrices...")
    if cp.exists('X_train.pkl') and cp.exists('y_train.pkl'):
        X_train = cp.load('X_train.pkl', verbose=False)
        y_train = cp.load('y_train.pkl', verbose=False)
        print(f"  ✓ Train resumed  {X_train.shape}")
    else:
        print("  Building train...")
        X_train, y_train = _build_xy(train_df, vectorizer)

    # ── Fit scaler on raw train features ──────────────────────────────────
    if cp.exists('scaler.pkl'):
        scaler = cp.load('scaler.pkl', verbose=False)
        X_train_s = scaler.transform(X_train)
    else:
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        cp.save(scaler, 'scaler.pkl')

    cp.save(X_train_s, 'X_train.pkl', verbose=False)
    cp.save(y_train,   'y_train.pkl', verbose=False)
    print(f"  ✓ Train  {X_train_s.shape}  labels: {np.bincount(y_train)}")

    # ── Validation ────────────────────────────────────────────────────────
    if cp.exists('X_val.pkl') and cp.exists('y_val.pkl'):
        X_val = cp.load('X_val.pkl', verbose=False)
        y_val = cp.load('y_val.pkl', verbose=False)
        print(f"  ✓ Val resumed    {X_val.shape}")
    else:
        print("  Building val...")
        X_val_raw, y_val = _build_xy(val_df, vectorizer)
        X_val = scaler.transform(X_val_raw)
        cp.save(X_val, 'X_val.pkl', verbose=False)
        cp.save(y_val, 'y_val.pkl', verbose=False)
        print(f"  ✓ Val    {X_val.shape}  labels: {np.bincount(y_val)}")

    # ── Test ──────────────────────────────────────────────────────────────
    if cp.exists('X_test.pkl') and cp.exists('y_test.pkl'):
        X_test = cp.load('X_test.pkl', verbose=False)
        y_test = cp.load('y_test.pkl', verbose=False)
        print(f"  ✓ Test resumed   {X_test.shape}")
    else:
        print("  Building test...")
        X_test_raw, y_test = _build_xy(test_df, vectorizer)
        X_test = scaler.transform(X_test_raw)
        cp.save(X_test, 'X_test.pkl', verbose=False)
        cp.save(y_test, 'y_test.pkl', verbose=False)
        print(f"  ✓ Test   {X_test.shape}  labels: {np.bincount(y_test)}")

    return X_train_s, y_train, X_val, y_val, X_test, y_test, vectorizer, scaler


# ============================================================================
# STEP 4 — SUPERVISED MODELS
# ============================================================================

def _evaluate(model, X, y, label, X_nb=False):
    """Return metrics dict and print a one-liner."""
    from sklearn.metrics import (accuracy_score, f1_score,
                                 precision_score, recall_score, confusion_matrix)
    y_pred = model.predict(X)
    acc  = accuracy_score(y, y_pred)
    f1   = f1_score(y, y_pred, zero_division=0)
    prec = precision_score(y, y_pred, zero_division=0)
    rec  = recall_score(y, y_pred, zero_division=0)
    cm   = confusion_matrix(y, y_pred)
    print(f"  ✓ Acc:{acc:.4f}  F1:{f1:.4f}  Prec:{prec:.4f}  Rec:{rec:.4f}")
    print(f"    CM: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    return dict(name=label, accuracy=float(acc), f1=float(f1),
                precision=float(prec), recall=float(rec),
                confusion_matrix=cm.tolist())


def train_supervised(X_train, y_train, X_val, y_val, X_test, y_test, cp: CheckpointManager):
    print("\n" + "=" * 70)
    print("STEP 4: SUPERVISED MODELS")
    print("=" * 70)

    from sklearn.linear_model    import LogisticRegression
    from sklearn.svm             import SVC
    from sklearn.naive_bayes     import MultinomialNB
    from sklearn.ensemble        import RandomForestClassifier

    results = {}

    # ── 1. Logistic Regression ────────────────────────────────────────────
    print("\n[1/4] Logistic Regression")
    try:
        if cp.exists('model_lr.pkl'):
            lr = cp.load('model_lr.pkl'); print("  (resumed)")
        else:
            lr = LogisticRegression(max_iter=1000, random_state=Config.RANDOM_SEED,
                                    n_jobs=-1, class_weight='balanced')
            lr.fit(X_train, y_train)
            cp.save(lr, 'model_lr.pkl')
        results['LR'] = _evaluate(lr, X_test, y_test, 'Logistic Regression')
    except Exception as e:
        print(f"  ✗ {e}")
        results['LR'] = {'name': 'Logistic Regression', 'error': str(e)}

    # ── 2. SVM ────────────────────────────────────────────────────────────
    print("\n[2/4] Support Vector Machine")
    try:
        if cp.exists('model_svm.pkl'):
            svm = cp.load('model_svm.pkl'); print("  (resumed)")
        else:
            svm = SVC(kernel='linear', C=1.0, random_state=Config.RANDOM_SEED,
                      class_weight='balanced')
            svm.fit(X_train, y_train)
            cp.save(svm, 'model_svm.pkl')
       # results['SVM'] = _evaluate(svm, X_test, y_test, 'SVM')
    except Exception as e:
        print(f"  ✗ {e}")
        results['SVM'] = {'name': 'SVM', 'error': str(e)}

    # ── 3. Naive Bayes ────────────────────────────────────────────────────
    print("\n[3/4] Naive Bayes")
    try:
        if cp.exists('model_nb.pkl'):
            nb = cp.load('model_nb.pkl'); print("  (resumed)")
        else:
            # NB needs non-negative; shift by per-set minimum
            X_nb = X_train - X_train.min() + 1e-10
            nb = MultinomialNB(alpha=1.0)
            nb.fit(X_nb, y_train)
            cp.save(nb,            'model_nb.pkl')
            cp.save(X_train.min(), 'nb_train_min.pkl', verbose=False)

        nb_min    = cp.load('nb_train_min.pkl', verbose=False) or X_train.min()
        X_test_nb = X_test - nb_min + 1e-10
        # clip any remaining negatives caused by val/test having a lower min
        X_test_nb = np.clip(X_test_nb, 1e-10, None)
        results['NB'] = _evaluate(nb, X_test_nb, y_test, 'Naive Bayes')
    except Exception as e:
        print(f"  ✗ {e}")
        results['NB'] = {'name': 'Naive Bayes', 'error': str(e)}

    # ── 4. Random Forest ─────────────────────────────────────────────────
    print("\n[4/4] Random Forest")
    try:
        if cp.exists('model_rf.pkl'):
            rf = cp.load('model_rf.pkl'); print("  (resumed)")
        else:
            rf = RandomForestClassifier(n_estimators=100, max_depth=15,
                                        random_state=Config.RANDOM_SEED,
                                        n_jobs=-1, class_weight='balanced')
            rf.fit(X_train, y_train)
            cp.save(rf, 'model_rf.pkl')
        results['RF'] = _evaluate(rf, X_test, y_test, 'Random Forest')
    except Exception as e:
        print(f"  ✗ {e}")
        results['RF'] = {'name': 'Random Forest', 'error': str(e)}

    return results




# ============================================================================
# STEP 5 — UNSUPERVISED / SEMI-SUPERVISED
# ============================================================================

def train_unsupervised(X_train, y_train, X_val, y_val, cp: CheckpointManager):
    print("\n" + "=" * 70)
    print("STEP 5: UNSUPERVISED / SEMI-SUPERVISED")
    print("=" * 70)

    from sklearn.cluster        import KMeans
    from sklearn.semi_supervised import LabelPropagation
    from sklearn.metrics        import (silhouette_score, adjusted_rand_score,
                                        accuracy_score, f1_score)

    results = {}

    # ── K-Means ───────────────────────────────────────────────────────────
    print("\n[1/2] K-Means (k=4)")
    try:
        if cp.exists('model_kmeans.pkl'):
            km = cp.load('model_kmeans.pkl'); print("  (resumed)")
            clusters = km.predict(X_train)
        else:
            km = KMeans(n_clusters=4, random_state=Config.RANDOM_SEED, n_init=10)
            clusters = km.fit_predict(X_train)
            cp.save(km, 'model_kmeans.pkl')

        sil = silhouette_score(X_train, clusters)
        ari = adjusted_rand_score(y_train, clusters)
        print(f"  ✓ Silhouette: {sil:.4f}  |  ARI: {ari:.4f}")
        results['KMeans'] = {'name': 'K-Means', 'silhouette': float(sil), 'ari': float(ari)}
    except Exception as e:
        print(f"  ✗ {e}")
        results['KMeans'] = {'name': 'K-Means', 'error': str(e)}

    # ── Label Propagation ─────────────────────────────────────────────────
    print("\n[2/2] Label Propagation (50% labeled)")
    try:
        if cp.exists('model_lp.pkl'):
            lp = cp.load('model_lp.pkl'); print("  (resumed)")
        else:
            rng     = np.random.default_rng(Config.RANDOM_SEED)
            y_semi  = y_train.copy()
            mask    = rng.choice(len(y_train), len(y_train) // 2, replace=False)
            y_semi[mask] = -1

            lp = LabelPropagation(kernel='rbf', gamma=0.1)
            lp.fit(X_train, y_semi)
            cp.save(lp, 'model_lp.pkl')

        acc = accuracy_score(y_val, lp.predict(X_val))
        f1  = f1_score(y_val, lp.predict(X_val), zero_division=0)
        print(f"  ✓ Val Acc: {acc:.4f}  |  Val F1: {f1:.4f}")
        results['LP'] = {'name': 'Label Propagation', 'accuracy': float(acc), 'f1': float(f1)}
    except Exception as e:
        print(f"  ✗ {e}")
        results['LP'] = {'name': 'Label Propagation', 'error': str(e)}

    return results


# ============================================================================
# STEP 6 — SUMMARY & SAVE RESULTS
# ============================================================================

def print_summary(sup: dict, unsup: dict):
    print("\n" + "=" * 70)
    print("FINAL RESULTS — MODEL A")
    print("=" * 70)

    print("\nSUPERVISED (test set):")
    print(f"  {'Model':<22} {'Acc':>8} {'F1':>8} {'Prec':>8} {'Rec':>8}")
    print("  " + "-" * 56)

    best_key, best_f1 = None, -1.0
    for k in ['LR', 'SVM', 'NB', 'RF']:
        r = sup.get(k, {})
        if 'error' in r:
            print(f"  {r['name']:<22}  ERROR: {r['error'][:35]}")
        else:
            print(f"  {r['name']:<22} {r['accuracy']:>8.4f} {r['f1']:>8.4f} "
                  f"{r['precision']:>8.4f} {r['recall']:>8.4f}")
            if r['f1'] > best_f1:
                best_f1, best_key = r['f1'], k

    if best_key:
        print(f"\n  ★ Best: {sup[best_key]['name']}  (F1 = {best_f1:.4f})")

    print("\nUNSUPERVISED / SEMI-SUPERVISED:")
    print("  " + "-" * 56)
    for r in unsup.values():
        if 'error' in r:
            print(f"  {r['name']:<22}  ERROR: {r['error'][:35]}")
        elif 'silhouette' in r:
            print(f"  {r['name']:<22}  Silhouette:{r['silhouette']:.4f}  ARI:{r['ari']:.4f}")
        else:
            print(f"  {r['name']:<22}  Acc:{r['accuracy']:.4f}  F1:{r['f1']:.4f}")


def save_results(sup, unsup, info, cp: CheckpointManager):
    payload = {
        'timestamp':           datetime.now().isoformat(),
        'supervised_models':   sup,
        'unsupervised_models': unsup,
        'dataset_info':        info,
    }
    cp.save(payload, 'model_a_results.pkl', verbose=False)

    results_path = os.path.join(cp.checkpoint_dir, 'model_a_results.json')
    with open(results_path, 'w') as fh:
        json.dump(payload, fh, indent=2)
    print(f"\n✓ Results JSON: {results_path}")


# ============================================================================
# MAIN
# ============================================================================

def run_pipeline():
    print("\n" + "=" * 70)
    print("MODEL A — ANSWER VERIFICATION PIPELINE")
    print("=" * 70)

    # 0. Mount Drive
    if not ensure_drive_mounted():
        return {'success': False, 'error': 'Drive not mounted'}

    cp = CheckpointManager(Config.CHECKPOINT_DIR)
    cp.status()

    try:
        # Steps 1–5
        df                                                         = load_data(cp)
        train_df, val_df, test_df                                  = split_data(df, cp)
        X_tr, y_tr, X_val, y_val, X_te, y_te, vectorizer, scaler  = preprocess_all(train_df, val_df, test_df, cp)
        sup_results                                                = train_supervised(X_tr, y_tr, X_val, y_val, X_te, y_te, cp)
        unsup_results                                              = train_unsupervised(X_tr, y_tr, X_val, y_val, cp)

        # Summary
        print_summary(sup_results, unsup_results)

        # Save
        dataset_info = {
            'total':      len(df),
            'train':      len(train_df),
            'val':        len(val_df),
            'test':       len(test_df),
            'n_features': int(X_tr.shape[1]),
        }
        save_results(sup_results, unsup_results, dataset_info, cp)

        print("\n" + "=" * 70)
        print("✅  MODEL A COMPLETE")
        print("=" * 70)
        cp.status()

        return {
            'success':    True,
            'cp':         cp,
            'vectorizer': vectorizer,
            'scaler':     scaler,
            'X_train': X_tr, 'y_train': y_tr,
            'X_val':   X_val, 'y_val':  y_val,
            'X_test':  X_te, 'y_test':  y_te,
        }

    except Exception as e:
        import traceback
        print(f"\n❌ PIPELINE FAILED: {e}")
        traceback.print_exc()
        return {'success': False, 'error': str(e)}


# ── entry point ───────────────────────────────────────────────────────────────
if __name__ == '__main__':
    out = run_pipeline()
    if out['success']:
        print("\n🎉 Done! Proceed to Model B.")
    else:
        print(f"\n❌ Error: {out.get('error')}")


MODEL A — ANSWER VERIFICATION PIPELINE
✓ Google Drive already mounted
✓ Checkpoint dir: /content/drive/MyDrive/.ipynb_checkpoints

Checkpoint directory: /content/drive/MyDrive/.ipynb_checkpoints
Files saved (21):
  .ipynb_checkpoints                  0.00 MB
  X_test.pkl                          122.26 MB
  X_train.pkl                         570.52 MB
  X_val.pkl                           122.26 MB
  data_test.pkl                       16.91 MB
  data_train.pkl                      78.92 MB
  data_val.pkl                        16.91 MB
  df_clean.pkl                        55.91 MB
  model_kmeans.pkl                    0.14 MB
  model_lr.pkl                        0.00 MB
  model_nb.pkl                        0.01 MB
  model_rf.pkl                        5.99 MB
  model_svm.pkl                       15.00 MB
  scaler.pkl                          0.01 MB
  split_test.pkl                      18.74 MB
  split_train.pkl                     49.54 MB
  split_val.pkl                      